<a href="https://colab.research.google.com/github/rounak393/clab/blob/main/encoder_attention_unet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

os.environ['KAGGLE_API_TOKEN'] = "KGAT_c03d989b55c966d18c971a92b023645b"

!kaggle datasets download -d britikak/busi-dataset



Dataset URL: https://www.kaggle.com/datasets/britikak/busi-dataset
License(s): unknown
100% 195M/195M [00:11<00:00, 17.2MB/s]



In [2]:
!unzip -q busi-dataset.zip -d busi_dataset

In [3]:
!pip install albumentations scikit-learn timm torchinfo ultralytics -q

import os
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import warnings

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from torchinfo import summary
from torchvision.models import resnet50, ResNet50_Weights

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ==========================================
# HYPERPARAMETERS & CONFIGURATION
# ==========================================
# IMPORTANT: Ensure this path points exactly to the folder containing 'benign' and 'malignant'
BASE_DIR = "/content/busi_dataset/Dataset_BUSI_with_GT"
CLASSES = ["benign", "malignant"]
IMG_SIZE = 256
BATCH_SIZE = 8
SEED = 42

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = True

# ==========================================
# DATASET & AUGMENTATIONS
# ==========================================
class BUSISegmentationDataset(Dataset):
    def __init__(self, base_dir, classes, transform=None):
        self.samples = []
        self.transform = transform

        for cls in classes:
            cls_dir = os.path.join(base_dir, cls)
            if not os.path.exists(cls_dir):
                continue

            images = [f for f in os.listdir(cls_dir) if f.endswith(".png") and "_mask" not in f]

            for img_name in images:
                img_path = os.path.join(cls_dir, img_name)
                base_name = img_name.replace(".png", "")
                mask_files = [f for f in os.listdir(cls_dir) if f.startswith(base_name + "_mask") and f.endswith(".png")]

                if len(mask_files) == 0:
                    continue

                mask_paths = [os.path.join(cls_dir, f) for f in mask_files]
                self.samples.append((img_path, mask_paths))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_paths = self.samples[idx]
        image = np.array(Image.open(img_path).convert("RGB"))

        combined_mask = np.zeros(image.shape[:2], dtype=np.uint8)
        for mpath in mask_paths:
            mask = np.array(Image.open(mpath).convert("L"))
            mask = (mask > 0).astype(np.uint8)
            combined_mask = np.logical_or(combined_mask, mask)

        combined_mask = combined_mask.astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=combined_mask)
            image = augmented["image"]
            combined_mask = augmented["mask"]

        combined_mask = (combined_mask > 0.5).astype(np.float32)
        image = torch.from_numpy(image).permute(2, 0, 1).float()
        mask = torch.from_numpy(combined_mask).unsqueeze(0).float()

        return image, mask


train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ElasticTransform(alpha=120, sigma=120 * 0.05, alpha_affine=120 * 0.03, p=0.5),
    A.GridDistortion(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), max_pixel_value=255.0)
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), max_pixel_value=255.0)
])

# Setup Loaders
full_dataset = BUSISegmentationDataset(BASE_DIR, classes=CLASSES, transform=None)
print(f"Total images found: {len(full_dataset)}")
assert len(full_dataset) > 0, "Dataset is empty! Check your BASE_DIR path to fix the num_samples=0 error."

indices = list(range(len(full_dataset)))
np.random.shuffle(indices)

train_size = int(0.8 * len(full_dataset))
val_size   = int(0.1 * len(full_dataset))

train_dataset = torch.utils.data.Subset(BUSISegmentationDataset(BASE_DIR, CLASSES, transform=train_transform), indices[val_size:train_size + val_size])
val_dataset = torch.utils.data.Subset(BUSISegmentationDataset(BASE_DIR, CLASSES, transform=val_transform), indices[:val_size])
test_dataset = torch.utils.data.Subset(BUSISegmentationDataset(BASE_DIR, CLASSES, transform=val_transform), indices[train_size + val_size:])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


# ==========================================
# M2 ABLATION MODEL: RESNET50-ONLY U-NET
# ==========================================
class DoubleConv(nn.Module):
    """Standard Convolutional Block without Attention"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class M2_ResNet50_UNet(nn.Module):
    """M2 Baseline: Standard ResNet50 U-Net (No Edges, No FastSAM, No Attention)"""
    def __init__(self):
        super().__init__()

        # 1. Standard ResNet50 Encoder (3-Channel Input)
        resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

        self.conv1   = resnet.conv1 # Standard 3-channel input
        self.bn1     = resnet.bn1
        self.relu    = resnet.relu
        self.maxpool = resnet.maxpool

        self.layer1  = resnet.layer1
        self.layer2  = resnet.layer2
        self.layer3  = resnet.layer3
        self.layer4  = resnet.layer4

        # 2. Standard Decoder (Upsampling + Concatenation + DoubleConv)
        # Bottleneck: 2048 channels (out of layer4)

        self.up4  = nn.ConvTranspose2d(2048, 1024, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(1024 + 1024, 1024) # e3 skip is 1024

        self.up3  = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(512 + 512, 512)   # e2 skip is 512

        self.up2  = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(256 + 256, 256)   # e1 skip is 256

        self.up1  = nn.ConvTranspose2d(256, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(64 + 64, 64)      # x0 skip is 64

        # Final upsample to match original image resolution
        self.up_out  = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec_out = DoubleConv(32, 32)
        self.final   = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        # --- ENCODER ---
        x0 = self.relu(self.bn1(self.conv1(x)))   # Skip 1 (64 channels)
        x_pool = self.maxpool(x0)

        e1 = self.layer1(x_pool)                  # Skip 2 (256 channels)
        e2 = self.layer2(e1)                      # Skip 3 (512 channels)
        e3 = self.layer3(e2)                      # Skip 4 (1024 channels)
        e4 = self.layer4(e3)                      # Bottleneck (2048 channels)

        # --- DECODER ---
        d4 = self.up4(e4)
        if d4.shape[-2:] != e3.shape[-2:]:
            d4 = F.interpolate(d4, size=e3.shape[-2:], mode='bilinear', align_corners=False)
        d4 = self.dec4(torch.cat([d4, e3], dim=1))

        d3 = self.up3(d4)
        if d3.shape[-2:] != e2.shape[-2:]:
            d3 = F.interpolate(d3, size=e2.shape[-2:], mode='bilinear', align_corners=False)
        d3 = self.dec3(torch.cat([d3, e2], dim=1))

        d2 = self.up2(d3)
        if d2.shape[-2:] != e1.shape[-2:]:
            d2 = F.interpolate(d2, size=e1.shape[-2:], mode='bilinear', align_corners=False)
        d2 = self.dec2(torch.cat([d2, e1], dim=1))

        d1 = self.up1(d2)
        if d1.shape[-2:] != x0.shape[-2:]:
            d1 = F.interpolate(d1, size=x0.shape[-2:], mode='bilinear', align_corners=False)
        d1 = self.dec1(torch.cat([d1, x0], dim=1))

        out = self.dec_out(self.up_out(d1))

        return self.final(out)


# ==========================================
# LOSS & METRICS
# ==========================================
class HybridLoss(nn.Module):
    def __init__(self, smooth=1.0, focal_gamma=2.0, alpha=0.3, beta=0.7):
        super().__init__()
        self.smooth = smooth
        self.focal_gamma = focal_gamma
        self.alpha = alpha
        self.beta = beta

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets)
        probs = torch.sigmoid(logits).view(-1)
        tgt = targets.view(-1)

        tp = (probs * tgt).sum()
        fp = (probs * (1.0 - tgt)).sum()
        fn = ((1.0 - probs) * tgt).sum()
        tversky = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
        tversky_loss = 1.0 - tversky

        pt = torch.where(tgt == 1, probs, 1.0 - probs)
        focal = (-(1.0 - pt) ** self.focal_gamma * torch.log(pt + 1e-8)).mean()

        return 0.4 * bce + 0.4 * tversky_loss + 0.2 * focal

def dice_coef(y_true, y_pred, smooth=1e-5):
    y_true_f, y_pred_f = y_true.view(-1), y_pred.view(-1)
    inter = (y_true_f * y_pred_f).sum()
    return (2. * inter + smooth) / (y_true_f.sum() + y_pred_f.sum() + smooth)

def iou_score(preds, masks, threshold=0.5, eps=1e-6):
    preds_bin, masks_bin = (preds > threshold).float(), (masks > threshold).float()
    inter = (preds_bin * masks_bin).sum().item()
    union = preds_bin.sum().item() + masks_bin.sum().item() - inter
    return inter / (union + eps)


# ==========================================
# M2 TRAINING LOOP
# ==========================================
def train_model_m2(model, train_loader, val_loader, epochs=50):
    model = model.to(device)
    criterion = HybridLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, min_lr=1e-6)

    best_val_loss = float("inf")
    best_model_weights = None

    for epoch in range(epochs):
        model.train()
        train_loss = train_dice = train_iou = 0
        for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            images, masks = images.to(device), masks.to(device)

            # Standard forward pass (No deep supervision tuple unpacking)
            preds_logits = model(images)
            loss = criterion(preds_logits, masks)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            preds_probs = torch.sigmoid(preds_logits)
            train_loss += loss.item()
            train_dice += dice_coef(masks, preds_probs).item()
            train_iou += iou_score(preds_probs, masks)

        model.eval()
        val_loss = val_dice = val_iou = 0
        with torch.no_grad():
            for images, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                images, masks = images.to(device), masks.to(device)
                preds_logits = model(images)
                loss = criterion(preds_logits, masks)

                preds_probs = torch.sigmoid(preds_logits)
                val_loss += loss.item()
                val_dice += dice_coef(masks, preds_probs).item()
                val_iou += iou_score(preds_probs, masks)

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss   = val_loss / len(val_loader)
        avg_train_dice = train_dice / len(train_loader)
        avg_val_dice   = val_dice / len(val_loader)

        print(f"\nTrain Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        print(f"Train Dice: {avg_train_dice:.4f} | Val Dice: {avg_val_dice:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_weights = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), "best_m2_resnet_unet.pth")
            print(f"✓ Best M2 model saved (Epoch {epoch+1}, Val Loss {best_val_loss:.4f})")

        scheduler.step(avg_val_loss)

    model.load_state_dict(best_model_weights)
    return model

# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
    model = M2_ResNet50_UNet().to(device)

    print("\n================= M2 MODEL SUMMARY =================\n")
    print(summary(model, input_size=(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE), mode='eval'))

    trained_m2 = train_model_m2(model, train_loader, val_loader, epochs=100)

    # Evaluate M2 on Test Set
    trained_m2.eval()
    test_dice, test_iou = 0, 0
    with torch.no_grad():
        for images, masks in tqdm(test_loader, desc="Evaluating M2 on Test Set"):
            images, masks = images.to(device), masks.to(device)
            preds_probs = torch.sigmoid(trained_m2(images))
            preds_bin = (preds_probs > 0.4).float()
            test_dice += dice_coef(masks, preds_bin).item()
            test_iou += iou_score(preds_probs, masks)

    num_batches = len(test_loader)
    print("\n" + "="*40)
    print("M2 TEST SET EVALUATION")
    print("="*40)
    print(f"M2 Dice Coefficient: {test_dice / num_batches:.4f}")
    print(f"M2 IoU (Jaccard):    {test_iou / num_batches:.4f}")
    print("="*40)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 65.2 MB/s eta 0:00:00
Using device: cuda
Total images found: 647
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 196MB/s]



================= M2 MODEL SUMMARY =================

Layer (type:depth-idx)                   Output Shape              Param #
M2_ResNet50_UNet                         [8, 1, 256, 256]          --
├─Conv2d: 1-1                            [8, 64, 128, 128]         9,408
├─BatchNorm2d: 1-2                       [8, 64, 128, 128]         128
├─ReLU: 1-3                              [8, 64, 128, 128]         --
├─MaxPool2d: 1-4                         [8, 64, 64, 64]           --
├─Sequential: 1-5                        [8, 256, 64, 64]          --
│    └─Bottleneck: 2-1                   [8, 256, 64, 64]          --
│    │    └─Conv2d: 3-1                  [8, 64, 64, 64]           4,096
│    │    └─BatchNorm2d: 3-2             [8, 64, 64, 64]           128
│    │    └─ReLU: 3-3                    [8, 64, 64, 64]           --
│    │    └─Conv2d: 3-4                  [8, 64, 64, 64]           36,864
│    │    └─BatchNorm2d: 3-5             [8, 64, 64, 64]           128
│    │    └─ReLU:

Epoch 1/100 [Val]: 100%|██████████| 8/8 [00:01<00:00,  6.37it/s]



Train Loss: 0.5113 | Val Loss: 0.4288
Train Dice: 0.2317 | Val Dice: 0.3355
✓ Best M2 model saved (Epoch 1, Val Loss 0.4288)


...


Train Loss: 0.0607 | Val Loss: 0.1196
Train Dice: 0.8934 | Val Dice: 0.8503


Epoch 100/100 [Val]: 100%|██████████| 8/8 [00:01<00:00,  4.58it/s]



Train Loss: 0.0586 | Val Loss: 0.1234
Train Dice: 0.8971 | Val Dice: 0.8472


Evaluating M2 on Test Set: 100%|██████████| 9/9 [00:05<00:00,  1.78it/s]


M2 TEST SET EVALUATION
M2 Dice Coefficient: 0.7590
M2 IoU (Jaccard):    0.6322
